## Pytorch is your friend 🤗

1. Tensors -> data structure
2. Automatic differentiation engine -> backprop
3. Deep learning framework -> training and inference

### Answer! 🤔

? What is a tensor?

? What is autograd?

? Efficient data loading? это как?

### Tensors

We use them because they efficiently represent data and operations on it. 

And we can quickly move data from CPU to GPU.

### DL framework

1. Pre-trained models
2. Loss functions
3. Optimizers.

In [2]:
import torch

torch.__version__
torch.cuda.is_available()

True

In [3]:
# Some basic operations with tensors

# tensor0d = torch.tensor(1)
# print(tensor0d.dtype)
# torchvec =tensor0d.to(torch.bfloat16)
# print(torchvec.dtype)
# print(torchvec)

# tensor0d_float = torch.tensor(1.0)
# print(tensor0d_float.dtype)

# tensor1d = torch.tensor([1,2,3])
# print(tensor1d.dtype)

tensor2d = torch.tensor([[1,2,3], [4,5,6]])
# print(tensor2d)
# print(tensor2d.T)

# What is the point of .reshape? it will create a wrong order hmm
tensor2d = tensor2d.reshape(1,1,3,2)
print(tensor2d.shape)
# x = tensor2d.reshape(-1) 
# print(x)



# tensor3d = torch.tensor([[[1,2,3], [4,5,6], [7,8,9]]])
# print(tensor3d)

torch.Size([1, 1, 3, 2])


In [ ]:
# understand the matrix multiplication first

tensor2d_1 = torch.tensor([[1,2,3], [4,5,6]])

tensor2d_2 = torch.tensor([[1,2,3], [4,5,6]])

tensor2d_1 @ tensor2d_2.T # tran
# [1, 2, 3]
# [4, 5, 6]

# [1, 4]
# [2, 5]
# [3, 6]

# r[1, 2, 3] * c[1, 2, 3]
# r[1, 2, 3] * c[4, 5, 6]

# r[4, 5, 6] * c[1, 2, 3]
# r[4, 5, 6] * c[4, 5, 6]

# [14, 32]
# [32, 77]

tensor([[14, 32],
        [32, 77]])

In [4]:
tensor2d_2.T

tensor([[1, 4],
        [2, 5],
        [3, 6]])

In [ ]:
import torch.nn.functional as F
from torch.autograd import grad

a = torch.tensor([0.9], requires_grad=True)
y = torch.tensor([1.0])

loss = F.binary_cross_entropy(a, y)
print(loss)
grad_L_a = grad(loss, a)[0]
print(grad_L_a)

# if gradient > 0, weight will ⬆️ decrease
# if gradient < 0, weight will ⬇️ increase
# gradient pushes weights in the opposite direction of itself

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = w1 * x1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

print(loss)

grad_L_w = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w)
print(grad_L_b)

loss.backward()

tensor(0.1054, grad_fn=<BinaryCrossEntropyBackward0>)
tensor([-1.1111])
tensor(0.0852, grad_fn=<BinaryCrossEntropyBackward0>)
(tensor([-0.0898]),)
(tensor([-0.0817]),)


### Multilayer perceptron (MLP)

In [6]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs),
        )
    
    def forward(self, x):
        logits = self.layers(x)
        return logits

torch.manual_seed(77714791777)

model = NeuralNetwork(num_inputs=50, num_outputs=3)
model.to(torch.bfloat16)
# model.layers[0].weight
# model.layers[0].weight.dtype

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)

In [8]:
some_shape = (1, 50)
x = torch.randn(some_shape)
x = x.to(torch.bfloat16)
out = model(x)
print(out)

# when we don't want to compute and save gradients
with torch.no_grad():
    out = model(x) 
print(out)

# apply softmax to the output
out = torch.softmax(out, dim=1)
print(out)


tensor([[-0.1621,  0.1758,  0.1777]], dtype=torch.bfloat16,
       grad_fn=<AddmmBackward0>)
tensor([[-0.1621,  0.1758,  0.1777]], dtype=torch.bfloat16)
tensor([[0.2617, 0.3691, 0.3691]], dtype=torch.bfloat16)


### Data loader

In [11]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
     ])

print(X_train.shape)
y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6]
])

y_test = torch.tensor([0, 1])

torch.Size([5, 2])


In [13]:
x_temp = X_train[0]
x_temp 

tensor([-1.2000,  3.1000])

In [14]:
from torch.utils.data import Dataset 

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X 
        self.labels = y
    
    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y
    
    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)
print(len(train_ds))

x0, y0 = train_ds[0]

# x0
x0
# y0

5


tensor([-1.2000,  3.1000])

In [17]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-1.2000,  3.1000],
        [ 2.3000, -1.1000]]) tensor([0, 1])
Batch 2: tensor([[-0.5000,  2.6000],
        [-0.9000,  2.9000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


In [19]:
# In practice, having a substantially smaller batch as the 
# last batch in a training epoch can disturb the convergence 
# during training. To prevent this, it’s recommended to set drop_last=True,
# which will drop the last batch in each epoch, as shown below:
# drop_last=True
train_loader = DataLoader( 
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[-0.5000,  2.6000],
        [-1.2000,  3.1000]]) tensor([0, 0])
Batch 2: tensor([[ 2.7000, -1.5000],
        [-0.9000,  2.9000]]) tensor([1, 0])


### Actual training loop

In [ ]:
import torch.nn.functional as F
import torch.nn as nn
import torch
from torch.utils.data import DataLoader, Dataset

class MLP_neural_network(nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(num_inputs, 30),
            nn.ReLU(),

            nn.Linear(30, 20),
            nn.ReLU(),

            nn.Linear(20, num_outputs)
        )
    
    def forward(self, x):
        logits = self.layers(x)
        return logits

torch.manual_seed(1477741)

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X 
        self.labels = y

    def __getitem__(self, index):
        out_x = self.features[index]
        out_y = self.labels[index]
        return out_x, out_y 
    
    def __len__(self):
        return self.labels.shape[0]
    
# toy data 
# train_data
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.6, 2.1],
    [2.1, -2.1],
    [3.1, -1.4]
])
y_train = torch.tensor([0, 0, 0, 1, 1])

# test_data  
# shift + option + arrow_up or arrow_down  
#  
#  
#  
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6]
])
y_test = torch.tensor([0, 1])

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    drop_last=True
)


test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=2
)


model = MLP_neural_network(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

NUM_EPOCHS = 3
for epoch in range(NUM_EPOCHS):
    
    model.train() # set model to training mode
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)

        loss = F.cross_entropy(logits, labels) # loss function

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ### LOGS
        print(
            f"Epoch {epoch+1:03d}/{NUM_EPOCHS:03d}"
            f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
            f" | Train/Val Loss: {loss:.2f}"
        )

    model.eval()

Epoch 001/003 | Batch 000/002 | Train/Val Loss: 1.02
Epoch 001/003 | Batch 001/002 | Train/Val Loss: 1.85
Epoch 002/003 | Batch 000/002 | Train/Val Loss: 0.07
Epoch 002/003 | Batch 001/002 | Train/Val Loss: 0.03
Epoch 003/003 | Batch 000/002 | Train/Val Loss: 0.02
Epoch 003/003 | Batch 001/002 | Train/Val Loss: 0.00


In [4]:
import torch

temp_1 = torch.tensor([[ 3.2846, -3.1264],
        [ 2.9755, -2.8467],
        [ 2.0955, -2.1111],
        [-2.9865,  1.8815],
        [-2.8769,  1.7912]])
pred = torch.argmax(temp_1, dim=1)
print(pred)
# tensor([0, 0, 0, 1, 1])

tensor([0, 0, 0, 1, 1])


In [2]:
model.eval()

with torch.no_grad():
    outputs = model(X_train)

print(outputs)

predictions = torch.argmax(outputs, dim=1)
print(predictions)

# torch.sum(predictions == y_test)

tensor([[ 3.2846, -3.1264],
        [ 2.9755, -2.8467],
        [ 2.0955, -2.0311],
        [-2.9865,  1.8815],
        [-2.8769,  1.7912]])
tensor([0, 0, 0, 1, 1])


In [35]:
from transformers import AutoModelForCausalLM

model_g = AutoModelForCausalLM.from_pretrained("google/gemma-3-4b-it")

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.26it/s]


In [ ]:
ef = examples_features = torch.tensor([2.0, -0.5])
mL1 = model_weights_0L = model.layers[0].weight
wL1 = model.layers[0].bias
out_linear = ef @ mL1.T + wL1
print(out_linear) # -> Formula inside the model is (x @ w.T + b)
ra = torch.relu(out_linear)
print(ra)

# model.layers[1](out_linear)
# mL2 = model.layers[2]
# wL2 = 



tensor([-0.4227, -1.1018, -0.1979, -4.0171,  6.1021,  5.4498, -4.9131,  5.9007,
        -2.7470, -0.0115], grad_fn=<AddBackward0>)
tensor([0.0000, 0.0000, 0.0000, 0.0000, 6.1021, 5.4498, 0.0000, 5.9007, 0.0000,
        0.0000], grad_fn=<ReluBackward0>)


In [22]:
# weights_before update
model.layers[0].weight

Parameter containing:
tensor([[-0.5161,  0.5740],
        [ 0.0646, -0.3645],
        [-0.3658, -0.0743],
        [ 0.4098, -0.0638],
        [ 0.4745,  0.0204],
        [ 0.0731, -0.5700],
        [-0.5840,  0.4129],
        [ 0.1856,  0.1267],
        [-0.1907, -0.0924],
        [ 0.3153, -0.3672]], requires_grad=True)

In [ ]:
x = torch.tensor([[0.1, 0.5]])
labels = torch.tensor([0])
logits = model(x)
loss = F.cross_entropy(logits, labels)
optimizer.zero_grad()
loss.backward() # that will only create the gradients
optimizer.step()
# grad defined based on the loss.
# Wi = Wi - lr * grad
# Bi = Bi - lr * grad


In [41]:
grads_of_layer_1 = model.layers[0].weight.grad
bias_grads_of_layer_1 = model.layers[0].bias.grad
print(bias_grads_of_layer_1)
model.layers[0].weight

tensor([ 0.0000,  0.0000,  0.0000,  0.0000, -0.0154,  0.0000,  0.0000, -0.0204,
         0.0000, -0.0110])


Parameter containing:
tensor([[-0.5161,  0.5740],
        [ 0.0646, -0.3645],
        [-0.3658, -0.0743],
        [ 0.4098, -0.0638],
        [ 0.4829,  0.0620],
        [ 0.0731, -0.5700],
        [-0.5840,  0.4129],
        [ 0.1945,  0.1713],
        [-0.1907, -0.0924],
        [ 0.3174, -0.3570]], requires_grad=True)

In [35]:
x = torch.tensor([[1.0, 2.0, 3.0],])
x_2 = torch.tensor([[1.0, 2.0, 3.0],
                  [1.0, 2.0, 3.0]])
# x_t = torch.tensor([[-0.5295,  0.6235, ])
print(x)
print(x.T)
print(x)
print(x @ x_2.T)

tensor([[1., 2., 3.]])
tensor([[1.],
        [2.],
        [3.]])
tensor([[1., 2., 3.]])
tensor([[14., 14.]])


In [ ]:
x = torch.tensor([[0.1, 0.5]])
# значит что мы умноажем 0.1 и 0.5 на каждый из этих вектров.
x_t = torch.tensor([[-0.5295,  0.6235], # -0.05295 + 0.31175
        [-0.4754, -0.4820],
        [ 0.7037,  0.3126],
        [-0.5638,  0.6699],
        [-0.1481,  0.1003],
        [-0.1595, -0.3440],
        [-0.0048,  0.4915],
        [-0.4054,  0.5695],
        [-0.7043,  0.5585],
        [-0.5226,  0.4490]])
print(x)
# print(x_t)
print(x_t.T)
print(x @ x_t.T)

tensor([[0.1000, 0.5000]])
tensor([[-0.5295, -0.4754,  0.7037, -0.5638, -0.1481, -0.1595, -0.0048, -0.4054,
         -0.7043, -0.5226],
        [ 0.6235, -0.4820,  0.3126,  0.6699,  0.1003, -0.3440,  0.4915,  0.5695,
          0.5585,  0.4490]])
tensor([[ 0.2588, -0.2885,  0.2267,  0.2786,  0.0353, -0.1880,  0.2453,  0.2442,
          0.2088,  0.1722]])


In [28]:
x = torch.tensor([[0.1, 0.5]])
x_t = torch.tensor([[-0.5295,  0.6235],
        [-0.4754, -0.4820],
        [ 0.7037,  0.3126],
        [-0.5638,  0.6699],
        [-0.1481,  0.1003],
        [-0.1595, -0.3440],
        [-0.0048,  0.4915],
        [-0.4054,  0.5695],
        [-0.7043,  0.5585],
        [-0.5226,  0.4490]])
print(x)
print(x_t)
print(x_t.T)
print(x @ x_t.T)

tensor([[0.1000, 0.5000]])
tensor([[-0.5295,  0.6235],
        [-0.4754, -0.4820],
        [ 0.7037,  0.3126],
        [-0.5638,  0.6699],
        [-0.1481,  0.1003],
        [-0.1595, -0.3440],
        [-0.0048,  0.4915],
        [-0.4054,  0.5695],
        [-0.7043,  0.5585],
        [-0.5226,  0.4490]])
tensor([[-0.5295, -0.4754,  0.7037, -0.5638, -0.1481, -0.1595, -0.0048, -0.4054,
         -0.7043, -0.5226],
        [ 0.6235, -0.4820,  0.3126,  0.6699,  0.1003, -0.3440,  0.4915,  0.5695,
          0.5585,  0.4490]])
tensor([[ 0.2588, -0.2885,  0.2267,  0.2786,  0.0353, -0.1880,  0.2453,  0.2442,
          0.2088,  0.1722]])


In [36]:
x = torch.tensor([[0.1, 0.5]])  # (1, 2)
print(x)
weight = model.layers[0].weight  # (10, 2)
print(weight)
bias = model.layers[0].bias      # (10,)
print("bais", bias)

output = x @ weight.T + bias  # (1, 10)

print("out",output)

tensor([[0.1000, 0.5000]])
Parameter containing:
tensor([[-0.5295,  0.6235],
        [-0.4754, -0.4820],
        [ 0.7037,  0.3126],
        [-0.5638,  0.6699],
        [-0.1481,  0.1003],
        [-0.1595, -0.3440],
        [-0.0048,  0.4915],
        [-0.4054,  0.5695],
        [-0.7043,  0.5585],
        [-0.5226,  0.4490]], requires_grad=True)
bais Parameter containing:
tensor([ 0.2068, -0.7005,  0.6329,  0.2891,  0.6309,  0.5242, -0.6127,  0.3601,
        -0.0094, -0.5742], requires_grad=True)
out tensor([[ 0.4656, -0.9891,  0.8596,  0.5677,  0.6662,  0.3362, -0.3674,  0.6044,
          0.1994, -0.4019]], grad_fn=<AddBackward0>)


## LLM from scratch

In [1]:
from transformers import AutoModelForCausalLM

model_g = AutoModelForCausalLM.from_pretrained("google/gemma-3-4b-it")
model_g


/home/vladimir_albrekht/miniconda3/envs/llm_from_scratch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.35it/s]


Gemma3ForConditionalGeneration(
  (model): Gemma3Model(
    (vision_tower): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(4096, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (layer_norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
              (self_attn): SiglipAttention(
                (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (out_proj): Linear(in_features=1152, out_features=1152, bias=True)
              )
              (layer_norm2): LayerNorm((1152,), eps=1e-06, elementwi

In [1]:
from transformers import AutoTokenizer 
import torch
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-4b-it")
tokenizer.vocab_size

data = [
    "Hello",
    "Anime",
    "World"
]

for i in range(len(data)):
    encoded = tokenizer.encode(data[i])
    print(encoded)
    decoded = tokenizer.decode(encoded)
    print(decoded)

print(data[1])

from torch.optim import Optimizer

# class MyCustomOptimizer(Optimizer):
#     def __init__(self, params, lr=1e-3):
#         defaults = dict(lr=lr)
#         super().__init__(params, defaults)

#     @torch.no_grad()
#     def step(self, closure=None):
#         for group in self.param_groups:
#             for p in group['params']:
#                 if p.grad is not None:
#                     p -= group['lr'] * p.grad

# optimizer_2 = MyCustomOptimizer

/home/vladimir_albrekht/miniconda3/envs/llm_from_scratch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2, 9259]
<bos>Hello
[2, 175511]
<bos>Anime
[2, 14447]
<bos>World
Anime


In [3]:
tokenizer.vocab_size

262144

In [10]:
x = torch.randn(3, 4)
print(x)
def _norm(x):
    return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6)

out = _norm(x)
print(out)

tensor([[ 0.7060, -1.5624, -0.8536,  0.8454],
        [ 1.7088,  0.8094, -1.7990,  0.9332],
        [-0.2012, -0.4912, -1.3939,  1.2696]])
tensor([[ 0.6744, -1.4926, -0.8155,  0.8077],
        [ 1.2330,  0.5841, -1.2981,  0.6733],
        [-0.2054, -0.5016, -1.4232,  1.2964]])


In [ ]:

import torch.nn as nn

weight = nn.Parameter(torch.zeros(256))

In [8]:
weight.shape

torch.Size([256])

## Main stuff 🩵

#### Your mom will be proud of you 🐈‍⬛

In [ ]:
import torch 
import torch.nn as nn
import torch.nn.functional as F

from transformers import Gemma3ForCausalLM
from transformers import PreTrainedModel, PretrainedConfig

from torch.utils.data import Dataset, DataLoader

# Any model is:
# PretrainedConfig
# PreTrainedModel
# Blocks:
#   - Attention
#   - MLP 
#   - RMSNorm
#   - Rotary Embedding

# Basically please use the architecture from transformers and write a working model that we can easily load and train.

# class KitanConfig(PretrainedConfig):
# and etc..

In [ ]:
# (language_model): Gemma3TextModel(
#     (embed_tokens): Gemma3TextScaledWordEmbedding(262208, 2560, padding_idx=0)
#     (layers): ModuleList(
#     (0-33): 34 x Gemma3DecoderLayer(
#         (self_attn): Gemma3Attention(
#         (q_proj): Linear(in_features=2560, out_features=2048, bias=False)
#         (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
#         (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
#         (o_proj): Linear(in_features=2048, out_features=2560, bias=False)
#         (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
#         (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
#         )
#         (mlp): Gemma3MLP(
#         (gate_proj): Linear(in_features=2560, out_features=10240, bias=False)
#         (up_proj): Linear(in_features=2560, out_features=10240, bias=False)
#         (down_proj): Linear(in_features=10240, out_features=2560, bias=False)
#         (act_fn): PytorchGELUTanh()
#         )
#         (input_layernorm): Gemma3RMSNorm((2560,), eps=1e-06)
#         (post_attention_layernorm): Gemma3RMSNorm((2560,), eps=1e-06)
#         (pre_feedforward_layernorm): Gemma3RMSNorm((2560,), eps=1e-06)
#         (post_feedforward_layernorm): Gemma3RMSNorm((2560,), eps=1e-06)
#     )
#     )
#     (norm): Gemma3RMSNorm((2560,), eps=1e-06)
#     (rotary_emb): Gemma3RotaryEmbedding()
#     (rotary_emb_local): Gemma3RotaryEmbedding()
# )

### Notes 📓 

In [ ]:
# about crossentropy loss
import torch 
import torch.nn.functional as F
logits = torch.tensor([
    [3.0, 1.0, 0.2, 0.1, 1.5, 5.2, 5.5, 1.0],  # метка = 0
])
labels = torch.tensor([2])
loss = F.cross_entropy(logits, labels, reduction='mean')
print(loss)

# something argmax

x = torch.tensor([[1.0, 4.2],
                 [2.0, 1.2]
                 ])
print(x)
print(torch.argmax(x, dim=0)) # columns

print(torch.argmax(x, dim=1)) # rows

# count params

# for i, p in enumerate(model.parameters()):
#     print(model.layers[i])
#     print(p.numel())
#     if i == len(model.layers) - 1:
#         break

def count_params(model):
    num_params = 0
    # print(model)
    for p in model.parameters():
        if p.requires_grad:
            num_params += p.numel()
    
    for i, p in enumerate(model.parameters()):
        print(model.layers[i])
        print(p.numel())
        if i == len(model.layers) - 1:
            break
    

    print(num_params)
    

count_params(model)

for p in model.layers[0].parameters():
    p.requires_grad = False

print(f"\n\nAfter freezing{'=' * 50}")
count_params(model)

# how to use super()

class Parent:
    def __init__(self):
        print("Parent __init__ called")
        self.value = 42

class Child(Parent):
    def __init__(self):
        super().__init__()
        print("Child __init__ called")

c = Child()
print(c.value)

class Parent:
    def __init__(self):
        print("Parent __init__ called")
        self.value = 42

    def print_value(self):
        print(self.value)

class Child(Parent):
    def __init__(self):
        # No super().__init__()
        self.value = 100
        print("Child __init__ called")

c = Child()
# print(c.value)
c.print_value()


# generator expression
some_text = "hello, world!"
print(sum(1 for char in some_text if char == "l"))

num_params = sum(
    print(p.numel()) for p in model.parameters() if p.requires_grad# generator expression
    # -
)
print("Total number of trainable model parameters:", num_params)